# Knowledge-distillation data for the health chatbot (teacher -> student)
Pipeline: result cards (from `capstone_classifier.ipynb`) -> questions -> **teacher LLM** writes answers -> automatic validation -> SFT files for the student (Qwen2.5-0.5B / 1.5B).

**Colab steps**
1. Upload `data/final/kd/cards_train.json` to `MyDrive/capstone_kd/` (once). Delete any old `kd_raw_*.jsonl` there if the validator changed since it was written.
2. Runtime -> Change runtime type -> **T4 GPU** (only needed for `TEACHER='qwen'`).
3. Run all. The first run with `qwen` installs vLLM; if it asks, restart the session and run all again. If the model fails to start, restart the session before retrying (a crashed engine keeps GPU memory).
4. Section 8 prints progress per batch and can be stopped (and re-run) at any time; finished items are kept (resume). `N_CARDS=30` is the quick pilot, 1500 the full run (about 3-4 h on a T4). Read the report in section 9 before building the SFT files.

The student must see exactly the same card text as in production, so `card_text()` (section 2) is part of the contract.

## 0. Config

In [ ]:
TEACHER = 'qwen'      # 'qwen' = local vLLM on the Colab T4 | 'gemini' | 'openrouter' | 'mock' (dry run, no GPU, no key)
N_CARDS = 1500        # pilot: 30 | full run: 1500 (x Q_PER_CARD = 3000 items; measured ~1000 items/hour with the 14B teacher on a T4)
Q_PER_CARD = 2
MAX_TRY = 2           # a failed answer is regenerated once, then dropped
SEED = 0
BATCH = 64            # items per checkpoint
MIN_PASS = 0.6        # safety stop: if the pass rate is below this after 128+ items, stop instead of burning hours

QWEN_MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'          # fallback if out of memory or too slow: 'Qwen/Qwen2.5-7B-Instruct-AWQ'
GEMINI_MODEL = 'gemini-2.5-flash'                      # UNVERIFIED name: check the current free-tier model in Google AI Studio
OPENROUTER_MODEL = 'qwen/qwen-2.5-72b-instruct:free'   # UNVERIFIED name: check openrouter.ai/models?q=free
API_SLEEP = 4.5       # seconds between API calls (free-tier rate limits)

## 1. Setup

In [ ]:
import os, json, re, random, time, collections
import numpy as np, pandas as pd

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive'); WORK = '/content/drive/MyDrive/capstone_kd'
else:
    WORK = 'kd_work'
os.makedirs(WORK, exist_ok=True)
CARDS_PATH = f'{WORK}/cards_train.json' if IN_COLAB else 'data/final/kd/cards_train.json'
assert os.path.exists(CARDS_PATH), f'upload cards_train.json to {CARDS_PATH}'

if TEACHER == 'qwen':
    try:
        import vllm
    except ImportError:
        get_ipython().system('pip -q install vllm')
        print('vLLM installed. Restart the session (Runtime -> Restart session) and run all again.')
print('teacher:', TEACHER, '| work dir:', WORK)

## 2. Cards and the card text shown to the models
`card_text()` turns a card into short plain text (easier for a 0.5B model than JSON). The same function must be used on the Raspberry Pi.

In [ ]:
DIS = ['hipertensi', 'diabetes', 'kolesterol_tinggi']
DIS_NAME = {'hipertensi': 'Hipertensi', 'diabetes': 'Diabetes', 'kolesterol_tinggi': 'Kolesterol tinggi'}
MISS_NAME = {'age': 'umur', 'male': 'jenis kelamin', 'bmi': 'BMI', 'sbp': 'tekanan darah sistolik', 'dbp': 'tekanan darah diastolik',
             'smoker': 'status merokok', 'waist': 'lingkar pinggang', 'hdl': 'HDL', 'fam_dm': 'riwayat diabetes keluarga',
             'fam_mi': 'riwayat serangan jantung keluarga', 'vig_rec': 'olahraga berat', 'mod_rec': 'olahraga sedang'}

def pct_text(s):
    return 'kurang dari 5%' if s == '<5%' else 'lebih dari 95%' if s == '>95%' else f'sekitar {s}'

def card_text(c):
    p, k = c['pasien'], c['kategori']
    fmt = lambda v: str(v).replace('.', ',')
    w = 'tidak diketahui' if p['lingkar_pinggang_cm'] is None else f"{fmt(p['lingkar_pinggang_cm'])} cm ({k['lingkar_pinggang']})"
    L = [f"Pasien: {p['jenis_kelamin']}, {p['umur']} tahun",
         f"BMI: {fmt(p['bmi'])} ({k['bmi']})",
         f"Tekanan darah: {p['td_sistolik']}/{p['td_diastolik']} mmHg ({k['tekanan_darah']})",
         f"Lingkar pinggang: {w}",
         f"HDL: {fmt(p['hdl_mgdl'])} mg/dL ({k['hdl']})",
         f"Pernah merokok: {p['pernah_merokok']}; riwayat diabetes keluarga: {p['riwayat_diabetes_keluarga']}; "
         f"riwayat serangan jantung keluarga: {p['riwayat_serangan_jantung_keluarga']}",
         f"Olahraga rekreasi: berat {p['olahraga_berat_rekreasi']}, sedang {p['olahraga_sedang_rekreasi']}",
         'Perkiraan risiko:']
    L += [f"- {DIS_NAME[d]}: {pct_text(c['risiko'][d]['persen_perkiraan'])} (tingkat {c['risiko'][d]['tingkat']})" for d in DIS]
    if c['data_tidak_lengkap']: L.append('Data tidak diisi: ' + ', '.join(MISS_NAME.get(m, m) for m in c['data_tidak_lengkap']))
    if c['peringatan']: L.append('Peringatan: ' + '; '.join(c['peringatan']))
    return '\n'.join(L)

cards = json.load(open(CARDS_PATH)); CID = {c['id_pasien']: c for c in cards}
print(len(cards), 'cards;', sum(bool(c['peringatan']) for c in cards), 'with warnings;', sum(bool(c['data_tidak_lengkap']) for c in cards), 'with missing data')
print(card_text(cards[0]))

## 3. Which cards (diversity)
Weighted random order: rare risk patterns are favoured (weight = 1/sqrt(pattern count)); every 30th slot is a card with a warning (blood-pressure red flag). The order is fixed by `SEED`, so raising `N_CARDS` extends the pilot instead of replacing it.

In [ ]:
pat = [tuple(c['risiko'][d]['tingkat'] for d in DIS) for c in cards]
cnt = collections.Counter(pat); w = np.array([1 / np.sqrt(cnt[p]) for p in pat])
rng = np.random.RandomState(SEED)
order = list(np.argsort(-(rng.random_sample(len(cards)) ** (1 / w))))
warn_idx = [i for i in order if cards[i]['peringatan']]; rest = [i for i in order if not cards[i]['peringatan']]
seq, wi, ri = [], 0, 0
while ri < len(rest) or wi < len(warn_idx):
    if len(seq) % 30 == 29 and wi < len(warn_idx): seq.append(warn_idx[wi]); wi += 1
    elif ri < len(rest): seq.append(rest[ri]); ri += 1
    else: seq.append(warn_idx[wi]); wi += 1
sel = [cards[i] for i in seq[:N_CARDS]]
print(len(sel), 'cards selected; patterns:', collections.Counter(tuple(c['risiko'][d]['tingkat'] for d in DIS) for c in sel).most_common(5), '... warnings:', sum(bool(c['peringatan']) for c in sel))

## 4. Question bank
Every card gets one *core* question (explain the card) and one *other* question (advice, doctor, hypothetical, or a question the bot must refuse). `hint` is shown only to the teacher.

In [ ]:
REFUSE = r'tidak (?:dapat|bisa|boleh|mampu|menghitung|mengubah|mendiagnosis|memberi|menilai|menyarankan|merekomendasikan)|belum (?:bisa|dapat)|di luar|hanya (?:bisa|dapat|mencakup|menjelaskan|membantu)'
Q = {   # type: (templates, teacher hint)
 'jelaskan': (['Tolong jelaskan hasil saya.', 'Hasilnya gimana?', 'Bisa jelaskan arti kartu hasil ini?', 'Saya kurang paham hasilnya, tolong dijelaskan dengan sederhana.', 'Ringkas hasil saya dong.'],
              'Ringkas kartu: tekanan darah, BMI, lingkar pinggang, HDL, lalu ketiga risiko dengan tingkatnya. Soroti hal yang paling perlu perhatian.'),
 'kenapa_tingkat': (['Kenapa risiko {d} saya tingkat {t}?', 'Apa yang bikin risiko {d} saya masuk tingkat {t}?'],
              'Kaitkan dengan faktor di kartu yang umumnya berhubungan dengan penyakit itu. Jangan klaim sebab pasti; ini perkiraan statistik.'),
 'arti_persen': (['Apa maksudnya {p} untuk {d}?', 'Berarti saya {p} kena {d}?'],
              'Jelaskan arti angka itu sebagai perkiraan kelompok orang dengan profil serupa, bukan kepastian.'),
 'arti_kategori': (['Apa maksud tekanan darah saya "{k_td}"?', 'BMI saya masuk "{k_bmi}", itu artinya apa?', 'Apa artinya HDL saya {k_hdl}?'],
              'Jelaskan arti kategori itu dengan bahasa sederhana, sesuai nilai di kartu.'),
 'gaya_hidup': (['Apa yang bisa saya lakukan untuk menurunkan risiko {d}?', 'Ada saran pola makan atau olahraga buat saya?', 'Kebiasaan apa yang sebaiknya saya ubah?'],
              'Beri saran gaya hidup umum (makan, aktivitas fisik, tidur, rokok) yang relevan dengan kartu. Tanpa obat, tanpa dosis, tanpa angka selain yang ada di kartu.'),
 'ke_dokter': (['Apakah saya perlu ke dokter?', 'Kapan sebaiknya saya periksa?'],
              'Sarankan pemeriksaan ke dokter atau fasilitas kesehatan, lebih mendesak bila ada tingkat tinggi atau peringatan.'),
 'akurasi': (['Seberapa akurat hasil ini?', 'Bisa dipercaya nggak hasil ini?'],
              'Jelaskan: perkiraan statistik dari data survei dewasa Amerika, bukan diagnosis, bisa meleset. Jangan menyebut angka akurasi.'),
 'data_kosong': (['Kenapa ada data yang kosong, apa pengaruhnya?', 'Data saya ada yang tidak terisi, apakah hasilnya tetap bisa dipakai?'],
              'Sebut data yang tidak diisi (lihat kartu), jelaskan hasil tetap perkiraan tetapi bisa kurang tepat; sarankan melengkapi data.'),
 'hipotetis': (['Kalau berat badan saya turun 5 kg, risikonya jadi berapa?', 'Kalau saya olahraga rutin, risiko saya turun berapa persen?', 'Kalau tekanan darah saya turun, risikonya berubah jadi apa?'],
              'Katakan tidak bisa menghitung ulang risiko untuk kondisi yang belum terjadi; boleh bicara arah umum tanpa angka baru; sarankan memasukkan data terbaru nanti.'),
 'obat': (['Obat apa yang harus saya minum untuk {d}?', 'Berapa dosis obat {d} yang cocok buat saya?', 'Boleh saya berhenti minum obat saya?'],
              'TOLAK dengan sopan. Jangan sebut nama obat, golongan obat, atau dosis. Arahkan ke dokter atau apoteker.'),
 'diagnosis': (['Jadi saya pasti kena {d} ya?', 'Apakah saya sudah terkena diabetes?', 'Tolong diagnosis saya.'],
              'TOLAK mendiagnosis. Kartu hanya perkiraan risiko. Arahkan ke dokter untuk pemeriksaan.'),
 'darurat': (['Dada saya sakit dan sesak sekarang, gimana?', 'Saya pusing berat dan pandangan kabur, apa itu karena tekanan darah?'],
              'DARURAT: minta segera ke IGD atau layanan gawat darurat terdekat. Jangan menunggu kartu. Jangan menyebut nomor telepon.'),
 'luar_topik': (['Siapa presiden Indonesia?', 'Tolong buatkan puisi tentang hujan.', 'Ada resep nasi goreng yang enak?'],
              'TOLAK sopan: kamu hanya membantu menjelaskan kartu hasil kesehatan. Tawarkan menjelaskan kartu.'),
 'ubah_angka': (['Bisa ubah tekanan darah saya di kartu biar risikonya rendah?', 'Hitung ulang risiko saya kalau umur saya 30 tahun.'],
              'TOLAK mengubah atau menghitung ulang di sini. Jika data keliru, minta memasukkan ulang data yang benar lewat formulir.'),
 'penyakit_lain': (['Apakah saya berisiko kanker?', 'Bagaimana risiko stroke saya?'],
              'Katakan kartu hanya mencakup hipertensi, diabetes, kolesterol tinggi; tidak bisa menilai penyakit lain; sarankan ke dokter.')}
CORE = {'jelaskan': .35, 'kenapa_tingkat': .25, 'arti_persen': .20, 'arti_kategori': .20}
OTHER = {'gaya_hidup': .22, 'ke_dokter': .12, 'akurasi': .06, 'data_kosong': .06, 'hipotetis': .10, 'obat': .10,
         'diagnosis': .08, 'darurat': .05, 'luar_topik': .07, 'ubah_angka': .05, 'penyakit_lain': .06}
REFUSAL_TYPES = {'obat', 'diagnosis', 'luar_topik', 'ubah_angka', 'penyakit_lain', 'hipotetis'}

def make_question(c, typ, r):
    d = r.choice(DIS); t = c['risiko'][d]['tingkat']
    f = dict(d=DIS_NAME[d].lower(), t=t, p=pct_text(c['risiko'][d]['persen_perkiraan']).replace('sekitar ', ''),
             k_td=c['kategori']['tekanan_darah'], k_bmi=c['kategori']['bmi'], k_hdl=c['kategori']['hdl'])
    tpl = r.choice(Q[typ][0]); return tpl.format(**f), d

def pick(wd, r, skip=()):
    ks = [k for k in wd if k not in skip]; return r.choice(ks, p=np.array([wd[k] for k in ks]) / sum(wd[k] for k in ks))

def make_items(sel):
    items = []
    for c in sel:
        r = np.random.RandomState(c['id_pasien'] + SEED); rr = random.Random(c['id_pasien'] + SEED)
        types = [pick(CORE, r)]
        for _ in range(Q_PER_CARD - 1):
            types.append(pick(OTHER, r, skip=set(types) | ({'data_kosong'} if not c['data_tidak_lengkap'] else set())))
        for j, typ in enumerate(types):
            q, d = make_question(c, typ, rr)
            items.append(dict(key=f"{c['id_pasien']}_{j}", id_pasien=c['id_pasien'], tipe=typ, pertanyaan=q, penyakit=d))
    return items

items = make_items(sel); print(len(items), 'items;', collections.Counter(i['tipe'] for i in items).most_common())
for i in items[:4]: print('-', i['tipe'], '|', i['pertanyaan'])

## 5. Prompts and gold examples
`SYS_TEACHER` is for the teacher. The gold examples below were written by hand and are used as few-shot turns; they must pass the validator (checked in section 6). `SYS_STUDENT` is the short system prompt the student sees in training and on the Pi.

In [ ]:
SYS_TEACHER = """Kamu adalah asisten kesehatan berbahasa Indonesia yang HANYA menjelaskan "kartu hasil" seorang pengguna dan menjawab pertanyaan lanjutan tentangnya.
Aturan wajib:
1. Gunakan hanya angka yang ada di kartu. Jangan menghitung, mengubah, atau mengarang angka baru. Tingkat risiko (rendah/sedang/tinggi) harus sama persis dengan kartu.
2. Angka risiko adalah perkiraan statistik dari data survei dewasa Amerika (NHANES), bukan diagnosis. Katakan "bukan diagnosis" dan anjurkan berkonsultasi dengan dokter atau tenaga kesehatan.
3. Jangan menyebut nama obat, golongan obat, atau dosis. Jangan mendiagnosis. Jangan memberi nomor telepon.
4. Jika ada keluhan darurat atau peringatan tekanan darah sangat tinggi di kartu, minta pengguna segera ke fasilitas kesehatan atau IGD.
5. Pertanyaan di luar penjelasan kartu (obat, diagnosis, topik lain, mengubah angka) ditolak dengan sopan, lalu tawarkan bantuan yang bisa kamu lakukan.
6. Tulis dalam bahasa Indonesia yang sederhana dan ramah, 60 sampai 170 kata, paragraf biasa tanpa judul atau markdown, dan selesaikan kalimat terakhir.
Jawab langsung dengan isi jawaban saja."""

SYS_STUDENT = ('Kamu asisten kesehatan yang hanya menjelaskan kartu hasil pasien. Gunakan hanya angka di kartu, jangan mendiagnosis, '
               'jangan menyebut nama obat atau dosis, selalu katakan ini bukan diagnosis dan anjurkan menemui dokter. Jawab dalam bahasa Indonesia.')

def user_msg(c, q): return f'KARTU HASIL:\n{card_text(c)}\n\nPERTANYAAN: {q}'

# gold cards are built by hand (values consistent with the rule engine)
def mk(sex, age, bmi, sbp, dbp, waist, hdl, smoker, fdm, fmi, risks, kat):
    return {'pasien': {'umur': age, 'jenis_kelamin': sex, 'bmi': bmi, 'td_sistolik': sbp, 'td_diastolik': dbp, 'pernah_merokok': smoker,
                       'lingkar_pinggang_cm': waist, 'hdl_mgdl': hdl, 'riwayat_diabetes_keluarga': fdm, 'riwayat_serangan_jantung_keluarga': fmi,
                       'olahraga_berat_rekreasi': 'tidak', 'olahraga_sedang_rekreasi': 'ya'},
            'kategori': kat, 'risiko': {d: {'persen_perkiraan': p, 'tingkat': t} for d, (p, t) in zip(DIS, risks)},
            'data_tidak_lengkap': [], 'peringatan': []}
GA = mk('laki-laki', 52, 27.8, 138, 88, 99, 38, 'ya', 'ya', 'tidak', [('55%', 'tinggi'), ('30%', 'tinggi'), ('40%', 'sedang')],
        {'tekanan_darah': 'hipertensi stage 1', 'bmi': 'obesitas I', 'lingkar_pinggang': 'di atas batas (obesitas sentral)', 'hdl': 'rendah'})
GB = mk('perempuan', 28, 21.5, 112, 70, 72, 62, 'tidak', 'tidak', 'tidak', [('<5%', 'rendah'), ('<5%', 'rendah'), ('10%', 'rendah')],
        {'tekanan_darah': 'normal', 'bmi': 'normal', 'lingkar_pinggang': 'normal', 'hdl': 'cukup'})
GOLD = [
 (GA, 'jelaskan', 'Tolong jelaskan hasil saya.',
  'Berikut ringkasan hasil Anda. Tekanan darah Anda 138/88 mmHg, masuk kategori hipertensi stage 1. BMI Anda 27,8 termasuk obesitas I dan lingkar pinggang 99 cm berada di atas batas, sedangkan HDL Anda 38 mg/dL tergolong rendah.\n\n'
  'Untuk perkiraan risiko: hipertensi sekitar 55% (tingkat tinggi), diabetes sekitar 30% (tingkat tinggi), dan kolesterol tinggi sekitar 40% (tingkat sedang). Angka ini adalah perkiraan statistik dari kelompok orang dengan profil serupa, bukan diagnosis.\n\n'
  'Riwayat diabetes dalam keluarga dan status pernah merokok ikut tercatat di kartu Anda. Sebaiknya hasil ini dibicarakan dengan dokter atau tenaga kesehatan untuk pemeriksaan lebih lanjut.'),
 (GB, 'arti_persen', 'Apa maksudnya 10% untuk kolesterol tinggi?',
  'Angka 10% berarti perkiraan risiko kolesterol tinggi Anda tergolong rendah. Secara kasar, dari 100 orang dengan profil seperti Anda, sekitar 10 orang diperkirakan pernah diberi tahu dokter bahwa kolesterolnya tinggi. '
  'Angka ini perkiraan statistik dan bukan diagnosis, jadi bukan berarti Anda pasti atau pasti tidak mengalaminya. Pemeriksaan laboratorium dan konsultasi dengan dokter tetap cara yang tepat untuk memastikan.'),
 (GA, 'obat', 'Obat apa yang harus saya minum untuk hipertensi?',
  'Saya tidak dapat menyarankan obat atau dosis. Pemilihan obat harus ditentukan dokter setelah memeriksa Anda secara langsung, karena bergantung pada kondisi dan riwayat kesehatan Anda. Kartu ini hanya perkiraan risiko, bukan diagnosis. '
  'Dengan tekanan darah 138/88 mmHg dan risiko hipertensi tingkat tinggi, sebaiknya Anda berkonsultasi dengan dokter atau tenaga kesehatan. Saya bisa membantu menjelaskan arti hasil Anda atau kebiasaan sehari-hari yang umumnya membantu menjaga tekanan darah.'),
 (GB, 'darurat', 'Dada saya sakit dan sesak sekarang, gimana?',
  'Nyeri dada dan sesak napas bisa menjadi tanda kondisi serius, jadi jangan menunggu hasil kartu ini. Segera hubungi layanan gawat darurat atau pergi ke IGD terdekat sekarang, dan minta orang di sekitar Anda untuk menemani. '
  'Kartu ini hanya perkiraan risiko jangka panjang dan bukan diagnosis, sehingga tidak bisa menilai keluhan yang sedang terjadi. Dokter atau tenaga kesehatan perlu memeriksa Anda secara langsung.'),
 (GA, 'hipotetis', 'Kalau berat badan saya turun 5 kg, risikonya jadi berapa?',
  'Saya tidak bisa menghitung ulang risiko untuk kondisi yang belum terjadi, karena angka di kartu hanya berlaku untuk data Anda saat ini. Secara umum, berat badan yang lebih sehat dapat membantu tekanan darah dan gula darah, tetapi besar perubahannya berbeda pada setiap orang. '
  'Jika nanti berat badan Anda benar-benar berubah, ukur ulang dan masukkan data terbaru untuk mendapatkan perkiraan baru. Hasil ini bukan diagnosis, dan dokter atau tenaga kesehatan bisa membantu menyusun target yang aman untuk Anda.')]
FEWSHOT = []
for g, typ, q, a in GOLD:
    FEWSHOT += [{'role': 'user', 'content': user_msg(g, q)}, {'role': 'assistant', 'content': a}]

RED = 'PERINGATAN KARTU: tekanan darah sangat tinggi. Awali jawaban dengan anjuran SEGERA ke fasilitas kesehatan atau IGD, lalu jawab pertanyaannya. '
def teacher_messages(c, it):
    red = RED if any('sangat tinggi' in x for x in c['peringatan']) else ''
    return ([{'role': 'system', 'content': SYS_TEACHER}] + FEWSHOT +
            [{'role': 'user', 'content': user_msg(c, it['pertanyaan']) + f"\n\n(Catatan untuk penulis jawaban, jangan disebut dalam jawaban: {red}{Q[it['tipe']][1]} Akhiri dengan satu kalimat bahwa ini perkiraan statistik, bukan diagnosis, dan sebaiknya berkonsultasi dengan dokter atau tenaga kesehatan.)"}])
print(len(GOLD), 'gold examples')

## 6. Automatic validation
Checks: not truncated, Indonesian, 30-230 words, numbers only from the card (plus a small whitelist of thresholds and list numbers), risk level words match the card (single-disease sentences only), disclaimer present, no drug names or doses, refusal types refuse, emergency answers send the user to the ER, red-flag cards say "segera". Known limits: it cannot check whether the *reasoning* is medically sound or whether a free-text answer is on topic (e.g. a lifestyle answer to a doctor question), and sentences naming two diseases skip the level check, so keep reading samples by hand.

In [ ]:
DRUGS = r'metformin|insulin|glibenklamid|glimepirid|gliklazid|amlodipin|amlodipine|captopril|kaptopril|lisinopril|ramipril|valsartan|losartan|candesartan|telmisartan|bisoprolol|atenolol|propranolol|furosemid|hidroklorotiazid|hctz|simvastatin|atorvastatin|rosuvastatin|statin|fenofibrat|aspirin|asetosal|warfarin|clopidogrel|paracetamol|ibuprofen|nifedipin|diltiazem|spironolakton|sulfonilurea|acarbose|sitagliptin|empagliflozin|dapagliflozin|liraglutide|semaglutide'
EN = {'the', 'and', 'you', 'your', 'is', 'are', 'of', 'to', 'with', 'for', 'that', 'this', 'risk', 'blood', 'pressure', 'please', 'doctor', 'should'}
ID = {'yang', 'dan', 'dengan', 'untuk', 'anda', 'saya', 'risiko', 'tidak', 'atau', 'ini', 'dari', 'ke', 'di', 'pada', 'dokter', 'adalah'}
WL = {1, 2, 3, 4, 5, 7, 10, 18.5, 23, 25, 30, 40, 50, 75, 80, 90, 100, 120, 130, 140, 150, 180}
DISW = {'hipertensi': ('hipertensi', 'tekanan darah tinggi', 'darah tinggi'), 'diabetes': ('diabetes', 'gula darah'), 'kolesterol_tinggi': ('kolesterol',)}

CARD_TYPES = {'jelaskan', 'kenapa_tingkat', 'arti_persen', 'arti_kategori'}

def own_numbers(c):   # the patient's own values: an answer that explains the card must cite at least one
    p = c['pasien']; a = {float(v) for v in (p['umur'], p['bmi'], p['td_sistolik'], p['td_diastolik'], p['lingkar_pinggang_cm'], p['hdl_mgdl']) if v is not None}
    return a | {float(x) for d in DIS for x in re.findall(r'\d+', c['risiko'][d]['persen_perkiraan'])}

def allowed_numbers(c):
    p, a = c['pasien'], set()
    for v in (p['umur'], p['bmi'], p['td_sistolik'], p['td_diastolik'], p['lingkar_pinggang_cm'], p['hdl_mgdl']):
        if v is not None: a |= {float(v), float(round(v)), round(float(v), 1)}
    for d in DIS:
        a |= {float(x) for x in re.findall(r'\d+', c['risiko'][d]['persen_perkiraan'])}
    return a | WL

def check(c, it, text, finish='stop'):
    bad, t = [], text.strip(); low = t.lower(); words = re.findall(r"[a-zA-Z']+", low)
    if finish == 'length' or not re.search(r'[.!?)\]"]$', t): bad.append('truncated')
    n = len(t.split())
    if not 30 <= n <= 230: bad.append(f'length:{n}')
    if sum(w in EN for w in words) >= 4 and sum(w in ID for w in words) < 2 * sum(w in EN for w in words): bad.append('not_indonesian')
    if sum(w in ID for w in words) < max(3, len(words) // 20): bad.append('not_indonesian')
    ok_num = allowed_numbers(c)
    extra = sorted({x for x in (float(m.replace(',', '.')) for m in re.findall(r'\d+(?:[.,]\d+)?', t)) if x not in ok_num})
    if extra: bad.append(f'numbers:{extra[:4]}')
    if it['tipe'] in CARD_TYPES and not ({float(m.replace(',', '.')) for m in re.findall(r'\d+(?:[.,]\d+)?', t)} & own_numbers(c)): bad.append('no_card_ref')
    sents = [s for s in re.split(r'(?<=[.!?])\s+|\n+', low) if s.strip()]
    for s in sents:
        found = [d for d, ks in DISW.items() if any(k in s for k in ks)]
        if len(found) != 1: continue
        s2 = re.sub(r'kolesterol tinggi|darah tinggi|tekanan darah tinggi|hdl rendah|lebih (?:rendah|tinggi)|menjadi (?:rendah|tinggi)', ' ', s)
        lv = re.findall(r'(?:risiko|tingkat)\b[^.!?\n]{0,40}?\b(rendah|sedang|tinggi)\b', s2)
        if any(x != c['risiko'][found[0]]['tingkat'] for x in lv): bad.append(f'level:{found[0]}'); break
    if not (re.search(r'bukan (?:sebuah )?diagnosis', low) and re.search(r'dokter|tenaga kesehatan|fasilitas kesehatan|igd', low)): bad.append('no_disclaimer')
    if re.search(DRUGS, low) or re.search(r'\d\s?(?:mg|mcg|ml|iu|unit)\b(?!\s?/\s?dl)', low): bad.append('drug_or_dose')
    if it['tipe'] in REFUSAL_TYPES and not re.search(REFUSE + (r'|hanya perkiraan|bukan diagnosis' if it['tipe'] == 'diagnosis' else ''), low): bad.append('no_refusal')
    if re.search(r'^\s*(?:[-*\u2022#]|\d+\.)\s', t, re.M): bad.append('markdown')          # plain paragraphs only (consistent style for the small student)
    if low.count('bukan diagnosis') > 1: bad.append('duplicate_disclaimer')
    if it['tipe'] == 'darurat' and not (re.search(r'segera', low) and re.search(r'igd|gawat darurat|darurat', low)): bad.append('no_emergency')
    if any('sangat tinggi' in x for x in c['peringatan']) and 'segera' not in low: bad.append('no_redflag')
    return bad

# tests: gold passes, tampered versions fail
_it = lambda typ: dict(tipe=typ)
for g, typ, q, a in GOLD: assert check(g, _it(typ), a) == [], (typ, check(g, _it(typ), a))
g, typ, q, a = GOLD[0]
assert 'numbers' in ' '.join(check(g, _it('jelaskan'), a.replace('138/88', '150/95')))
g3, _, _, a3 = GOLD[2]
assert 'level:hipertensi' in check(g3, _it('obat'), a3.replace('hipertensi tingkat tinggi', 'hipertensi tingkat rendah'))
assert 'no_card_ref' in check(g, _it('jelaskan'), 'Hasil Anda perlu diperhatikan dengan baik. Perkiraan ini bukan diagnosis, jadi sebaiknya Anda bicarakan dengan dokter atau tenaga kesehatan untuk pemeriksaan lanjutan ya.')
assert 'drug_or_dose' in check(g, _it('jelaskan'), a + ' Anda bisa minum metformin 500 mg.')
assert 'truncated' in check(g, _it('jelaskan'), a[:-30])
assert 'no_disclaimer' in check(g, _it('jelaskan'), a.replace('bukan diagnosis', 'sudah pasti'))
assert 'not_indonesian' in check(g, _it('jelaskan'), 'Your blood pressure is high and you should see the doctor for the check and the treatment of this risk. It is not a diagnosis, so please talk to your doctor soon.')
assert 'no_refusal' in check(g, _it('obat'), 'Gunakan obat apa saja yang Anda suka, ini bukan diagnosis. Tanyakan pada dokter jika ragu mengenai hasil kartu hasil Anda ini ya.')
assert 'markdown' in check(g, _it('jelaskan'), a.replace('Untuk perkiraan risiko:', 'Untuk perkiraan risiko:\n- hipertensi 55%'))
assert 'duplicate_disclaimer' in check(g, _it('jelaskan'), a + ' Ini perkiraan statistik, bukan diagnosis, dan sebaiknya berkonsultasi dengan dokter.')
assert 'no_refusal' not in check(GA, _it('diagnosis'), 'Kartu hasil Anda menunjukkan risiko diabetes sekitar 30% (tingkat tinggi), tetapi ini hanya perkiraan statistik dan bukan diagnosis. Sebaiknya Anda berkonsultasi dengan dokter atau tenaga kesehatan untuk pemeriksaan lebih lanjut.')
print('validator tests passed')

## 7. Teacher backends
`generate(batch)` takes a list of chat messages and returns `[(text, finish_reason)]`. API keys come from Colab Secrets (left key icon): `GEMINI_API_KEY` or `OPENROUTER_API_KEY`.

In [ ]:
def _secret(name):
    try: return userdata.get(name)
    except Exception: return os.environ.get(name)

_llm = None
def gen_qwen(batch, temp):
    global _llm
    import sys, types
    # Workaround (seen on Colab): vLLM's startup warmup imports a MiniMax-M3 module that needs `cutlass`, which is not installed -> engine crash.
    # Run the engine in this process (so the patch applies) and stub that irrelevant warmup out.
    os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
    _stub = types.ModuleType('vllm.model_executor.warmup.minimax_m3_msa_warmup'); _stub.minimax_m3_msa_warmup = lambda *a, **k: None
    sys.modules.setdefault('vllm.model_executor.warmup.minimax_m3_msa_warmup', _stub)
    from vllm import LLM, SamplingParams
    if _llm is None:
        _llm = LLM(QWEN_MODEL, dtype='half', max_model_len=6144, gpu_memory_utilization=0.90, enforce_eager=True)   # T4: no bf16, tight memory
    outs = _llm.chat(batch, SamplingParams(temperature=temp, top_p=0.9, max_tokens=450))
    return [(o.outputs[0].text, o.outputs[0].finish_reason) for o in outs]

def gen_gemini(batch, temp):
    import requests
    res = []
    for m in batch:
        sysm = next(x['content'] for x in m if x['role'] == 'system')
        contents = [{'role': 'user' if x['role'] == 'user' else 'model', 'parts': [{'text': x['content']}]} for x in m if x['role'] != 'system']
        r = requests.post(f'https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent', params={'key': _secret('GEMINI_API_KEY')},
                          json={'systemInstruction': {'parts': [{'text': sysm}]}, 'contents': contents,
                                'generationConfig': {'temperature': temp, 'topP': 0.9, 'maxOutputTokens': 450}}, timeout=120)
        try:
            cand = r.json()['candidates'][0]; res.append((cand['content']['parts'][0]['text'], 'length' if cand.get('finishReason') == 'MAX_TOKENS' else 'stop'))
        except Exception: res.append(('', f'error:{r.status_code}'))
        time.sleep(API_SLEEP)
    return res

def gen_openrouter(batch, temp):
    import requests
    res = []
    for m in batch:
        r = requests.post('https://openrouter.ai/api/v1/chat/completions', headers={'Authorization': 'Bearer ' + _secret('OPENROUTER_API_KEY')},
                          json={'model': OPENROUTER_MODEL, 'messages': m, 'temperature': temp, 'top_p': 0.9, 'max_tokens': 450}, timeout=120)
        try:
            ch = r.json()['choices'][0]; res.append((ch['message']['content'], ch.get('finish_reason') or 'stop'))
        except Exception: res.append(('', f'error:{r.status_code}'))
        time.sleep(API_SLEEP)
    return res

def gen_mock(batch, temp):   # dry run: random gold answers, so most fail the number/level checks (tests the pipeline, not quality)
    return [(random.choice(GOLD)[3], 'stop') for _ in batch]

GEN = {'qwen': gen_qwen, 'gemini': gen_gemini, 'openrouter': gen_openrouter, 'mock': gen_mock}[TEACHER]

## 8. Generate with checkpointing and resume
Each answer is appended to `kd_raw_<teacher>.jsonl` (on Drive in Colab). Re-running continues where it stopped; failed items get `MAX_TRY` attempts with a higher temperature.

In [ ]:
RAW = f'{WORK}/kd_raw_{TEACHER}.jsonl'
done = {}
if os.path.exists(RAW):
    for l in open(RAW):
        r = json.loads(l); done[r['key']] = r            # last record per key wins

def pending():
    return [it for it in items if not (it['key'] in done and (done[it['key']]['ok'] or done[it['key']]['tries'] >= MAX_TRY))]

CLOSING = ' Ini perkiraan statistik, bukan diagnosis; sebaiknya berkonsultasi dengan dokter atau tenaga kesehatan.'
def repair(txt, fin):   # the disclaimer is boilerplate: if it is the only thing missing, append it instead of discarding a good answer
    t = txt.strip()
    if fin == 'stop' and re.search(r'[.!?]$', t) and ('bukan diagnosis' not in t.lower() or not re.search(r'dokter|tenaga kesehatan', t.lower())): return t + CLOSING, True
    return t, False

t0, n_new = time.time(), 0
while pending():
    todo = pending()[:BATCH]
    tries = [done[it['key']]['tries'] if it['key'] in done else 0 for it in todo]
    outs = GEN([teacher_messages(CID[it['id_pasien']], it) for it in todo], temp=0.6 + 0.2 * max(tries))
    with open(RAW, 'a') as f:
        for it, k, (txt, fin) in zip(todo, tries, outs):
            txt, fixed = repair(txt, fin)
            bad = check(CID[it['id_pasien']], it, txt, fin) if txt.strip() else ['empty:' + fin]
            rec = dict(it, jawaban=txt.strip(), finish=fin, ok=not bad, alasan=bad, tries=k + 1, teacher=TEACHER, diperbaiki=fixed)
            done[it['key']] = rec; f.write(json.dumps(rec, ensure_ascii=False) + '\n'); n_new += 1
    rate = np.mean([r['ok'] for r in done.values()])
    print(f'{len(done)}/{len(items)} items | {n_new} new | {(time.time() - t0) / 60:.1f} min | pass so far {rate:.0%}')
    if len(done) >= 128 and rate < MIN_PASS: print(f'STOPPED: pass rate below {MIN_PASS:.0%}. Read the failures in section 9 before spending more time.'); break

## 9. Pilot report: pass rate, failure reasons, samples

In [ ]:
rows = [done[it['key']] for it in items if it['key'] in done]
df = pd.DataFrame(rows)
print(f'teacher={TEACHER}  items={len(df)}  pass={df.ok.mean():.1%}  mean words (passed)={df[df.ok].jawaban.str.split().str.len().mean():.0f}')
print('\npass rate by question type:'); print(df.groupby('tipe').ok.agg(['mean', 'count']).round(2).T.to_string())
print('\nfailure reasons:'); print(collections.Counter(a.split(':')[0] for al in df[~df.ok].alasan for a in al).most_common())
print('\nrepaired (closing sentence appended):', int(df['diperbaiki'].fillna(False).sum()) if 'diperbaiki' in df else 0, 'of', len(df))
if TEACHER != 'mock':
    for _, r in df[~df.ok].head(4).iterrows():
        print('\n--- FAILED', r.tipe, r.alasan, '|', r.pertanyaan); print('>>', r.jawaban)
    for _, r in df[df.ok].sample(min(3, int(df.ok.sum())), random_state=0).iterrows():
        print('\n---', r.tipe, '|', r.pertanyaan); print(card_text(CID[r.id_pasien])); print('>>', r.jawaban)

### Compare teachers (run the pilot once per teacher with the same `N_CARDS` and `SEED`)
Pick Qwen if its pass rate is close to Gemini/OpenRouter and the samples read fine in Indonesian. `kd_raw_*.jsonl` files from different teachers sit side by side.

In [ ]:
import glob
cmp = []
for p in sorted(glob.glob(f'{WORK}/kd_raw_*.jsonl')):
    d = pd.DataFrame([json.loads(l) for l in open(p)]).drop_duplicates('key', keep='last')
    cmp.append(dict(teacher=d.teacher.iloc[0], items=len(d), pass_rate=round(d.ok.mean(), 3), avg_words=round(d[d.ok].jawaban.str.split().str.len().mean())))
print(pd.DataFrame(cmp).to_string(index=False))

## 10. Build the student SFT files
Only validated answers. Split 90/10 **by patient**, so no patient appears in both files. With several teachers, the file of the chosen `TEACHER` is used; to mix teachers, concatenate the raw files first.

In [ ]:
ok = df[df.ok].copy()
ids = sorted(ok.id_pasien.unique()); rs = np.random.RandomState(SEED); rs.shuffle(ids)
val_ids = set(ids[:max(1, len(ids) // 10)])
def rec(r): return {'messages': [{'role': 'system', 'content': SYS_STUDENT}, {'role': 'user', 'content': user_msg(CID[r.id_pasien], r.pertanyaan)},
                                 {'role': 'assistant', 'content': r.jawaban}], 'id_pasien': int(r.id_pasien), 'tipe': r.tipe, 'teacher': r.teacher}
tr_rows = [rec(r) for r in ok.itertuples() if r.id_pasien not in val_ids]; va_rows = [rec(r) for r in ok.itertuples() if r.id_pasien in val_ids]
for name, rows in [('sft_train', tr_rows), ('sft_val', va_rows)]:
    with open(f'{WORK}/{name}.jsonl', 'w') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(len(tr_rows), 'train examples,', len(va_rows), 'val examples ->', WORK)

# stub for the human-written test set (~100 questions, never shown to the teacher): fill "pertanyaan" by hand
rt = np.random.RandomState(SEED + 1); held = [i for i in sorted(CID) if i not in set(ok.id_pasien)]; rt.shuffle(held)
stub = [{'id_pasien': int(i), 'kartu': card_text(CID[i]), 'pertanyaan': ''} for i in held[:100]]
json.dump(stub, open(f'{WORK}/test_prompts_TEMPLATE.json', 'w'), ensure_ascii=False, indent=1)
print('wrote test_prompts_TEMPLATE.json: 100 patients not used for training; write your own question for each')